# Model Training & Selection

This notebook compares Logistic Regression, Decision Tree, and Random
Forest using the same split-then-SMOTE procedure implemented in
`src/train_model.py`, so the comparison reflects what production actually
does. The Random Forest trained here via `src.train_model.train()` is the
identical artifact `main.py` saves to `../models/churn_random_forest.pkl`.

**What changed from the previous version of this notebook:** SMOTE used
to be applied to the *entire* dataset before any split (in
`02_preprocessing.ipynb`'s old version), which let synthetic minority rows
leak into the test set and inflated every model's reported metrics. Here,
the split happens first and SMOTE is applied only to the training fold —
expect the numbers below to be somewhat lower, but honest.

## 1. Imports

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
from imblearn.over_sampling import SMOTE

from src.feature_engineering import add_features
from src.data_preprocessing import preprocess_data
from src.train_model import train

import warnings
warnings.filterwarnings("ignore")

## 2. Rebuild the Production Feature Pipeline

Loaded directly from `src/`, identical to `02_preprocessing.ipynb` and to
what `main.py` runs.

In [ ]:
raw_df = pd.read_csv("../data/raw/customer_churn_raw.csv")
raw_df.columns = raw_df.columns.str.strip().str.replace(" ", "_")

engineered_df = add_features(raw_df)
X_processed, y, raw_features = preprocess_data(engineered_df)

print("X_processed shape:", X_processed.shape)
print(y.value_counts(normalize=True))

## 3. Shared Train/Test Split — BEFORE SMOTE

Same parameters as `src/train_model.py` (`test_size=0.2`, `stratify=y`,
`random_state=42`), so this split is identical to the one `train()`
performs internally — making the manual LR/DT comparison below and the
production RF model directly comparable on the same held-out customers.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42
)

## 4. SMOTE — Training Fold Only

Mirrors the `FIX (SMOTE leakage)` comment in `src/train_model.py`:
synthetic minority samples are generated only from `X_train` / `y_train`,
so `X_test` / `y_test` always reflect the real-world (imbalanced) class
distribution.

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
y_train_res.value_counts(normalize=True)

## 5. Baseline: Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_res, y_train_res)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, lr_pred))
print("Recall:", recall_score(y_test, lr_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr_prob))

## 6. Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=6, min_samples_split=50, random_state=42)
dt.fit(X_train_res, y_train_res)

dt_pred = dt.predict(X_test)
dt_prob = dt.predict_proba(X_test)[:, 1]

print("Decision Tree")
print("Accuracy:", accuracy_score(y_test, dt_pred))
print("Recall:", recall_score(y_test, dt_pred))
print("ROC-AUC:", roc_auc_score(y_test, dt_prob))

## 7. Random Forest — via `src.train_model.train` (Production Path)

Rather than re-implementing the Random Forest step a second time, this
calls the actual production function. It performs the identical
split-then-SMOTE steps as above internally, fits the model, and saves it
to `../models/churn_random_forest.pkl` — exactly what `main.py` does.

In [ ]:
rf_model, X_test_prod, y_test_prod = train(X_processed, y)

rf_pred = rf_model.predict(X_test_prod)
rf_prob = rf_model.predict_proba(X_test_prod)[:, 1]

print("Random Forest (production)")
print("Accuracy:", accuracy_score(y_test_prod, rf_pred))
print("Recall:", recall_score(y_test_prod, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test_prod, rf_prob))

## 8. Model Comparison Summary

Run the cells above and fill in the printed values — they're intentionally
not hard-coded here since the leakage fix means the previously published
numbers (≈0.88 accuracy / 0.94 ROC-AUC for Random Forest) no longer apply.

| Model               | Accuracy | Recall (Churn) | ROC–AUC |
|--------------------|----------|-----------------|---------|
| Logistic Regression | —        | —               | —       |
| Decision Tree       | —        | —               | —       |
| Random Forest       | —        | —               | —       |

### Expected pattern
- Logistic Regression should remain the weakest baseline, validating that
  preprocessing and feature engineering produced learnable signal.
- Decision Tree should improve on the linear baseline by capturing
  non-linear interactions.
- Random Forest should still come out ahead on recall and ROC–AUC, since
  ensembling reduces variance relative to a single tree — but absolute
  scores across all three models should now be lower than the old,
  leakage-inflated notebook reported, and that's expected and correct.

## 9. Conclusion

Random Forest remains the selected production model, consistent with
`main.py` / `src/train_model.py`. The fitted model and preprocessor are
already persisted to `../models/`. Continue to `05_evaluation.ipynb` for
detailed diagnostics (confusion matrix, ROC/PR curves, threshold tuning,
feature importance, and the business-facing revenue-at-risk summary).